# Laboratório — Cross-entropy, log-loss e perplexidade

Este laboratório acompanha a [Aula 22](../aulas/22-cross-entropy-perplexidade.md). Vamos implementar as métricas com NumPy, conferir estabilidade numérica e reproduzir armadilhas de avaliação.

**Ambiente recomendado:** Python 3.11+, NumPy 2.0+, pandas 2.0+ e Matplotlib 3.8+.  
**Reprodutibilidade:** seed fixa `20260908`; dados pequenos e sintéticos; nenhuma rede ou credencial.

In [ ]:
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 20260908
rng = np.random.default_rng(SEED)

print(f"Python {platform.python_version()}")
print(f"NumPy {np.__version__} | pandas {pd.__version__} | Matplotlib {matplotlib.__version__}")
print(f"Seed: {SEED}")

## 1. Funções com contratos explícitos

`cross_entropy` recebe duas distribuições. Se `p[k] > 0` e `q[k] = 0`, o custo é infinito. Para log-loss empírica, selecionamos a probabilidade da classe observada. Não aplicamos *clipping* silencioso.

In [ ]:
def validar_distribuicao(p, nome="p"):
    p = np.asarray(p, dtype=float)
    if p.ndim != 1 or p.size < 2:
        raise ValueError(f"{nome} deve ser vetor com pelo menos duas categorias")
    if not np.all(np.isfinite(p)) or np.any(p < 0):
        raise ValueError(f"{nome} contém valor inválido")
    if not np.isclose(p.sum(), 1.0, atol=1e-12):
        raise ValueError(f"{nome} deve somar 1")
    return p


def cross_entropy(p, q, base=np.e):
    p = validar_distribuicao(p, "p")
    q = validar_distribuicao(q, "q")
    if p.shape != q.shape:
        raise ValueError("p e q devem ter o mesmo suporte")
    if np.any((p > 0) & (q == 0)):
        return np.inf
    mask = p > 0
    return float(-np.sum(p[mask] * np.log(q[mask])) / np.log(base))


def log_loss_multiclasse(y, probabilidades):
    y = np.asarray(y, dtype=int)
    q = np.asarray(probabilidades, dtype=float)
    if q.ndim != 2 or len(y) != len(q):
        raise ValueError("shapes incompatíveis")
    if np.any(q < 0) or not np.allclose(q.sum(axis=1), 1.0, atol=1e-12):
        raise ValueError("cada linha deve ser uma distribuição")
    corretas = q[np.arange(len(y)), y]
    if np.any(corretas == 0):
        return np.inf
    return float(-np.log(corretas).mean())


print("H([0,8; 0,2], [0,7; 0,3]) =", cross_entropy([0.8, 0.2], [0.7, 0.3]))
print("Suporte impossível =", cross_entropy([0.8, 0.2], [1.0, 0.0]))

## 2. Entropia versus cross-entropy

A distribuição verdadeira é `p = [0.8, 0.2]`. Comparamos a previsão perfeita e dois modelos imperfeitos. A base natural produz nats.

In [ ]:
p = np.array([0.8, 0.2])
modelos = {
    "ideal p": p,
    "q_A": np.array([0.7, 0.3]),
    "q_B": np.array([0.4, 0.6]),
}

tabela_ce = pd.DataFrame([
    {"modelo": nome, "q": q.tolist(), "cross_entropy_nats": cross_entropy(p, q)}
    for nome, q in modelos.items()
])
print(tabela_ce.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

h_p = cross_entropy(p, p)
ce_a = cross_entropy(p, modelos["q_A"])
ce_b = cross_entropy(p, modelos["q_B"])
print()
print(f"H(p)={h_p:.9f}; excedente A={ce_a-h_p:.9f}; excedente B={ce_b-h_p:.9f}")

## 3. *One-hot* seleciona a classe correta

Em cada linha, a soma da cross-entropy tem apenas um termo não nulo. A perda é a NLL da classe observada.

In [ ]:
y_one_hot = np.array([0.0, 1.0, 0.0])
previsoes = {
    "cauteloso": np.array([0.25, 0.50, 0.25]),
    "confiante correto": np.array([0.01, 0.98, 0.01]),
    "confiante errado": np.array([0.98, 0.01, 0.01]),
}

linhas = []
for nome, q in previsoes.items():
    ce_one_hot = -np.sum(y_one_hot * np.log(q))
    nll_classe = -np.log(q[1])
    linhas.append({"previsão": nome, "q(classe B)": q[1], "loss": ce_one_hot})
    assert np.isclose(ce_one_hot, nll_classe)

print(pd.DataFrame(linhas).to_string(index=False, float_format=lambda x: f"{x:.6f}"))

## 4. Mesma acurácia, log-loss diferente

Os dois modelos abaixo erram o mesmo exemplo. O modelo B erra com 99% de confiança, portanto sua log-loss é muito pior.

In [ ]:
y = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])
q1_a = np.array([0.20, 0.30, 0.40, 0.45, 0.55, 0.55, 0.60, 0.70, 0.80, 0.90])
q1_b = np.array([0.01, 0.02, 0.10, 0.20, 0.99, 0.51, 0.80, 0.90, 0.98, 0.99])

def binario_para_matriz(q1):
    return np.column_stack([1 - q1, q1])


comparacao = []
for nome, q1 in {"A moderado": q1_a, "B extremo": q1_b}.items():
    pred = (q1 >= 0.5).astype(int)
    comparacao.append({
        "modelo": nome,
        "acurácia": np.mean(pred == y),
        "log_loss": log_loss_multiclasse(y, binario_para_matriz(q1)),
    })

comparacao = pd.DataFrame(comparacao)
print(comparacao.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

## 5. Binary cross-entropy e confiança

Para um exemplo positivo, a perda é `-log(q)`. Para um negativo, é `-log(1-q)`. O gráfico mostra a assimetria em torno do rótulo.

In [ ]:
def binary_cross_entropy(y, q):
    y = np.asarray(y, dtype=float)
    q = np.asarray(q, dtype=float)
    if np.any((q <= 0) | (q >= 1)):
        raise ValueError("use probabilidades estritamente entre 0 e 1")
    return -(y * np.log(q) + (1-y) * np.log1p(-q))


grade = np.linspace(0.001, 0.999, 500)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(grade, binary_cross_entropy(1, grade), label="rótulo y=1")
ax.plot(grade, binary_cross_entropy(0, grade), label="rótulo y=0")
ax.set(xlabel="Probabilidade prevista para Y=1", ylabel="Perda (nats)",
       title="Cross-entropy pune confiança na classe errada")
ax.set_ylim(0, 7)
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

print(f"y=1, q=0,9: {binary_cross_entropy(1, 0.9):.6f}")
print(f"y=1, q=0,1: {binary_cross_entropy(1, 0.1):.6f}")

## 6. Logits, estabilidade e gradiente

Implementamos `log_softmax` subtraindo o maior logit. Depois comparamos o gradiente analítico `q - y` com diferenças centrais.

In [ ]:
def log_softmax(logits):
    z = np.asarray(logits, dtype=float)
    m = np.max(z)
    return z - m - np.log(np.exp(z - m).sum())


def nll_logits(logits, classe):
    return float(-log_softmax(logits)[classe])


logits_extremos = np.array([1000.0, 999.0, -1000.0])
log_q = log_softmax(logits_extremos)
q = np.exp(log_q)
print("log-softmax estável:", log_q)
print("probabilidades:", q, "soma=", q.sum())

logits = np.array([1.2, -0.4, 0.7])
classe = 2
q_grad = np.exp(log_softmax(logits))
y_hot = np.eye(3)[classe]
grad_analitico = q_grad - y_hot

eps = 1e-6
grad_numerico = np.empty_like(logits)
for j in range(len(logits)):
    desloc = np.zeros_like(logits)
    desloc[j] = eps
    grad_numerico[j] = (
        nll_logits(logits + desloc, classe) - nll_logits(logits - desloc, classe)
    ) / (2 * eps)

erro_grad = np.max(np.abs(grad_analitico - grad_numerico))
print("gradiente analítico:", grad_analitico)
print("gradiente numérico: ", grad_numerico)
print(f"maior erro absoluto={erro_grad:.3e}")

## 7. Perplexidade de uma sequência

Somamos a NLL dos tokens observados, dividimos pelo número de alvos válidos e exponenciamos uma única vez.

In [ ]:
def perplexidade(probabilidades_corretas):
    probs = np.asarray(probabilidades_corretas, dtype=float)
    if probs.ndim != 1 or np.any((probs <= 0) | (probs > 1)):
        raise ValueError("probabilidades devem pertencer a (0, 1]")
    nll_media = -np.log(probs).mean()
    return float(np.exp(nll_media)), float(nll_media)


probs_seq = np.array([1/2, 1/4, 1/8])
ppl_seq, nll_seq = perplexidade(probs_seq)
print(f"Probabilidades={probs_seq.tolist()}")
print(f"NLL média={nll_seq:.9f} nat; PPL={ppl_seq:.9f}")
print(f"Inverso da média geométrica={1/np.exp(np.log(probs_seq).mean()):.9f}")

## 8. Tokenizadores diferentes: unidade diferente

A mesma probabilidade total `1/16` pode ser fatorada em dois ou quatro tokens. A probabilidade da sequência é igual, mas a média por token — e a PPL — muda.

In [ ]:
tok_a = np.array([1/4, 1/4])
tok_b = np.array([1/2, 1/2, 1/2, 1/2])

ppl_a, nll_a = perplexidade(tok_a)
ppl_b, nll_b = perplexidade(tok_b)
print(f"Produto A={tok_a.prod():.6f}; tokens={len(tok_a)}; NLL/token={nll_a:.6f}; PPL={ppl_a:.6f}")
print(f"Produto B={tok_b.prod():.6f}; tokens={len(tok_b)}; NLL/token={nll_b:.6f}; PPL={ppl_b:.6f}")
print("A probabilidade total coincide; a PPL por token não.")

## 9. Agregação por token

Não tire a média das perplexidades das sequências. Acumule NLL e tokens válidos, depois calcule a exponencial.

In [ ]:
sequencias = pd.DataFrame({
    "sequência": ["A", "B"],
    "tokens_válidos": [2, 8],
    "nll_total": [2.0, 16.0],
})
sequencias["nll_média"] = sequencias["nll_total"] / sequencias["tokens_válidos"]
sequencias["ppl"] = np.exp(sequencias["nll_média"])

nll_macro = sequencias["nll_média"].mean()
ppl_média_errada = sequencias["ppl"].mean()
nll_micro = sequencias["nll_total"].sum() / sequencias["tokens_válidos"].sum()
ppl_corpus = np.exp(nll_micro)

print(sequencias.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print()
print(f"Média simples das PPLs (não é PPL do corpus): {ppl_média_errada:.6f}")
print(f"NLL macro por sequência: {nll_macro:.6f}")
print(f"NLL micro por token: {nll_micro:.6f}; PPL do corpus: {ppl_corpus:.6f}")

## 10. Contexto truncado versus janela deslizante

Este exemplo sintético mantém os mesmos alvos, mas atribui probabilidades melhores quando cada posição recebe mais contexto. Ele ilustra por que protocolos de PPL devem fixar janela e estratégia; não representa um modelo treinado.

In [ ]:
probs_blocos = np.array([0.25, 0.30, 0.20, 0.35, 0.25, 0.30, 0.20, 0.35])
probs_deslizante = np.array([0.25, 0.30, 0.38, 0.42, 0.40, 0.45, 0.37, 0.43])

ppl_blocos, nll_blocos = perplexidade(probs_blocos)
ppl_deslizante, nll_deslizante = perplexidade(probs_deslizante)
print(f"Blocos independentes: NLL={nll_blocos:.6f}; PPL={ppl_blocos:.6f}")
print(f"Janela deslizante:   NLL={nll_deslizante:.6f}; PPL={ppl_deslizante:.6f}")

## 11. Verificações metodológicas

As asserções confirmam identidades matemáticas, estabilidade, gradiente, agregação e os fenômenos didáticos usados na aula.

In [ ]:
assert np.isclose(h_p, 0.5004024235381879)
assert np.isclose(ce_a, 0.5261345160161732)
assert np.isclose(ce_b, 0.8351977102525918)
assert comparacao.loc[0, "acurácia"] == comparacao.loc[1, "acurácia"] == 0.9
assert comparacao.loc[1, "log_loss"] > comparacao.loc[0, "log_loss"]
assert np.isclose(q.sum(), 1.0)
assert erro_grad < 1e-8
assert np.isclose(ppl_seq, 4.0)
assert np.isclose(tok_a.prod(), tok_b.prod())
assert np.isclose(ppl_a, 4.0) and np.isclose(ppl_b, 2.0)
assert np.isclose(nll_micro, 1.8)
assert not np.isclose(ppl_média_errada, ppl_corpus)
assert ppl_deslizante < ppl_blocos

print("Todas as verificações foram aprovadas.")
print(f"Cross-entropy: H(p)={h_p:.6f}; A={ce_a:.6f}; B={ce_b:.6f}")
print(f"Mesma acurácia=0,90; log-loss A={comparacao.loc[0, 'log_loss']:.6f}; B={comparacao.loc[1, 'log_loss']:.6f}")
print(f"Gradiente: erro máximo={erro_grad:.3e}")
print(f"Sequência: PPL={ppl_seq:.6f}")
print(f"Tokenização: mesma probabilidade total, PPL A={ppl_a:.1f} e B={ppl_b:.1f}")
print(f"Corpus: NLL/token={nll_micro:.6f}; PPL={ppl_corpus:.6f}")
print(f"Contexto: PPL blocos={ppl_blocos:.6f}; deslizante={ppl_deslizante:.6f}")

## Conclusões

- Cross-entropy compara a distribuição real com a prevista.
- Em rótulos *one-hot*, a loss é a NLL da classe correta.
- Acurácia igual pode esconder riscos probabilísticos muito diferentes.
- `log_softmax` estável evita *overflow*, e o gradiente é `q - y`.
- PPL é a exponencial da NLL média por token válido.
- Tokenizador, corpus, janela, máscara e denominador fazem parte da definição operacional da métrica.
- Para o corpus, some NLL e tokens antes de exponenciar.

Volte à [Aula 22](../aulas/22-cross-entropy-perplexidade.md) para o checklist, exercícios e referências.